## Executive Summary

**Purpose:** Every diameter, tooth thickness, contact ratio and inspection dimension of an external involute helical gear and of a meshing pair of them, with profile shift.

**What it does:** `HelicalGear` holds one gear (teeth, normal module, helix angle and hand, normal pressure angle, profile shift, basic rack, face width) and derives its transverse quantities, its diameters, the base helix angle, the lead and the virtual tooth count used for strength. `helical_mesh_check` solves a pair: operating transverse pressure angle, centre distance, transverse, overlap and total contact ratios, active profile, interference and tip clearance. The span measurement $W_k$ and the dimension over two balls are worked out in the normal section, as an inspector measures them.

**Method:** a helical gear is a spur gear in every transverse section, twisted along the axis. So each gear has an *equivalent transverse spur gear*: module $m_t$, pressure angle $\alpha_t$, shift and rack heights rescaled so that every length stays the same. That gear is a `gears.involute.GearGeometry`, and all the planar work (diameters, root form, mesh checks, tooth outlines) is reused from the spur module unchanged. The helix only adds the overlap ratio, the normal-section measurements and, in `gears.helical.rating`, the helix factors of ISO 6336.

**Units:** millimetres, newtons and MPa, as everywhere in `emachines.gears`. Angles are in radians in the code and in degrees in reports.

**Hand:** a right-hand gear has teeth that slope like a right-hand thread. The two gears of an external pair have opposite hands and the same helix angle.

## How It Fits Into the Helical Gear Workflow

**Upstream:** `gears.involute` (involute function, spur gear geometry and mesh checks, which this module reuses through the transverse equivalent).

**Downstream:** `gears.helical.layout` (tooth counts, helix angle, profile shift split), `gears.helical.rating` (ISO 6336 with helix factors), `gears.helical.manufacturing` (data sheets, tolerances, outlines), `gears.helical.design` (search and package).

In [ ]:
#| default_exp gears.helical.geometry

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass, replace

from emachines.gears.involute import (GearGeometry, inv, inv_inverse, mesh_check, operating_pressure_angle,
                                      centre_distance)

In [ ]:
#| hide
import numpy as np
import matplotlib.pyplot as plt

## Notation

| Symbol | Meaning | Units |
|:--:|:--------------------------------|:--:|
| $z$ | number of teeth | – |
| $m_n$ | normal module (the hob's module) | mm |
| $m_t$ | transverse module, $m_t = m_n/\cos\beta$ | mm |
| $\beta$ | helix angle on the reference cylinder | rad |
| $\beta_b$ | base helix angle | rad |
| $\alpha_n$ | normal pressure angle (the hob's pressure angle) | rad |
| $\alpha_t$ | transverse pressure angle | rad |
| $x$ | profile shift coefficient, in normal modules | – |
| $d$, $d_b$, $d_a$, $d_f$ | reference, base, tip and root diameters | mm |
| $s_n$, $s_t$ | tooth thickness on the reference circle, normal and transverse (arc) | mm |
| $p_x$ | axial pitch | mm |
| $p_z$ | lead (axial advance of one tooth per turn) | mm |
| $z_n$ | virtual tooth count | – |
| $b$ | face width | mm |
| $\alpha_{wt}$ | operating transverse pressure angle | rad |
| $a_w$ | operating centre distance | mm |
| $\varepsilon_\alpha$, $\varepsilon_\beta$, $\varepsilon_\gamma$ | transverse, overlap and total contact ratio | – |
| $W_k$ | span measurement over $k$ teeth, in the normal section | mm |
| $M_d$ | dimension over two balls of diameter $D_M$ | mm |

## Normal and Transverse Sections

The hob that cuts a helical gear is a rack in the **normal section**, perpendicular to the teeth. Its module $m_n$, pressure angle $\alpha_n$ and heights $h_{aP}^*m_n$, $h_{fP}^*m_n$ are the standard ones (ISO 53). In the **transverse section**, perpendicular to the gear axis, the same rack appears stretched along the pitch line by $1/\cos\beta$ and keeps its heights (ISO 21771):

$$m_t = \frac{m_n}{\cos\beta},\qquad \tan\alpha_t = \frac{\tan\alpha_n}{\cos\beta},\qquad \sin\beta_b = \sin\beta\cos\alpha_n .$$

The diameters follow from the transverse values. The tip and root are set by the normal heights and the shift in normal modules:

$$d = z\,m_t,\qquad d_b = d\cos\alpha_t,\qquad d_a = d + 2m_n(h_{aP}^* + x - k^*),\qquad d_f = d - 2m_n(h_{fP}^* - x).$$

The reference tooth thickness is $s_n = m_n(\pi/2 + 2x\tan\alpha_n)$ in the normal section and $s_t = s_n/\cos\beta$ in the transverse section.

The helix angle changes with diameter, because the lead $p_z = \pi d/\tan\beta$ is the same at every radius:

$$\tan\beta_y = \tan\beta\,\frac{d_y}{d} .$$

At the base cylinder this gives $\tan\beta_b = \tan\beta\cos\alpha_t$, which is the same as the sine relation above. The axial pitch is $p_x = \pi m_n/\sin\beta$.

### The equivalent transverse spur gear

Every planar question about the gear can be answered in one transverse section, where the flanks are ordinary involutes of the base circle $d_b$. The spur formulas of `gears.involute` take a module, a pressure angle, a shift and rack heights *per unit module*. Expressed in transverse modules, these keep every length the same:

$$m \to m_t,\quad \alpha \to \alpha_t,\quad x \to x\cos\beta,\quad h_{aP}^* \to h_{aP}^*\cos\beta,\quad h_{fP}^* \to h_{fP}^*\cos\beta,\quad k^* \to k^*\cos\beta .$$

Then $z\,m_t$, $2m_t(h_{aP}^*\cos\beta + x\cos\beta) = 2m_n(h_{aP}^* + x)$ and $m_t(\pi/2 + 2x\cos\beta\tan\alpha_t) = s_t$ all come out right.

The tool tip radius $\rho_{fP}$ is a circle in the normal section and an ellipse in the transverse section. The root form diameter depends only on where the straight flank ends, $m_n\,[h_{fP}^* - \rho_{fP}^*(1-\sin\alpha_n)]$ below the reference line. That height is the same in both sections. So the transverse radius coefficient is chosen to keep that height:

$$\rho_{fP,t}^* = \rho_{fP}^*\cos\beta\,\frac{1-\sin\alpha_n}{1-\sin\alpha_t}.$$

With that radius, `GearGeometry.d_form` and the undercut limit are exact for a hobbed helical gear. The fillet *shape* drawn by `gear_outline` is a circle of that radius, where the real fillet is an ellipse. The difference is a few hundredths of a module, which is fine for a layout or a DXF outline but not for a root stress FEA.

The undercut limit, for example, becomes

$$z_{min} = \frac{2\cos\beta\,[h_{fP}^* - x - \rho_{fP}^*(1-\sin\alpha_n)]}{\sin^2\alpha_t},$$

which is about 14 at $\beta = 15°$ against 17 for a spur gear without shift.

### The virtual gear

Tooth strength is worked out on the **virtual spur gear**: the spur gear whose tooth, in the normal section, matches the helical tooth. ISO 6336-3 uses

$$z_n = \frac{z}{\cos^2\beta_b\cos\beta},$$

with the normal module, the normal pressure angle and the same shift. `HelicalGear.virtual()` returns it as a `GearGeometry` with a non-integer tooth count, ready for `gears.rating.form_factors`.

In [ ]:
#| export
@dataclass(frozen=True)
class HelicalGear:
    """
    One external involute helical gear (ISO 21771). Lengths in mm, angles in radians.

    Attributes
    ----------
    z       : int    Number of teeth
    mn      : float  Normal module [mm]
    beta    : float  Helix angle on the reference cylinder [rad] (0 ≤ β < 45°; 0 gives a spur gear)
    x       : float  Profile shift coefficient, in normal modules
    b       : float  Face width [mm]
    hand    : str    'right' or 'left'
    alpha_n : float  Normal pressure angle [rad] (default 20°)
    ha, hf  : float  Basic rack addendum and dedendum coefficients (default 1.0, 1.25)
    rho_f   : float  Basic rack root radius coefficient (default 0.38, ISO 53 profile A)
    k_tip   : float  Tip reduction coefficient k*, in normal modules (default 0)
    name    : str    Label used in reports
    """
    z: int
    mn: float
    beta: float
    x: float = 0.0
    b: float = 10.0
    hand: str = "right"
    alpha_n: float = math.radians(20.0)
    ha: float = 1.0
    hf: float = 1.25
    rho_f: float = 0.38
    k_tip: float = 0.0
    name: str = "gear"

    def __post_init__(self):
        if self.z < 3:
            raise ValueError("z must be a tooth count ≥ 3")
        if self.mn <= 0 or self.b <= 0:
            raise ValueError("module and face width must be positive")
        if not 0 <= self.beta < math.radians(45):
            raise ValueError("helix angle must satisfy 0 ≤ β < 45°")
        if self.hand not in ("right", "left"):
            raise ValueError("hand must be 'right' or 'left'")

    # --- transverse quantities ---------------------------------------------------
    @property
    def mt(self) -> float:
        "Transverse module [mm]."
        return self.mn / math.cos(self.beta)

    @property
    def alpha_t(self) -> float:
        "Transverse pressure angle [rad]."
        return math.atan(math.tan(self.alpha_n) / math.cos(self.beta))

    @property
    def beta_b(self) -> float:
        "Base helix angle [rad]."
        return math.asin(math.sin(self.beta) * math.cos(self.alpha_n))

    # --- diameters -----------------------------------------------------------------
    @property
    def d(self) -> float:
        "Reference diameter [mm]."
        return self.z * self.mt

    @property
    def db(self) -> float:
        "Base diameter [mm]."
        return self.d * math.cos(self.alpha_t)

    @property
    def da(self) -> float:
        "Tip diameter [mm]."
        return self.d + 2 * self.mn * (self.ha + self.x - self.k_tip)

    @property
    def df(self) -> float:
        "Root diameter [mm]."
        return self.d - 2 * self.mn * (self.hf - self.x)

    @property
    def h(self) -> float:
        "Tooth depth [mm]."
        return (self.da - self.df) / 2

    # --- pitches -----------------------------------------------------------------------
    @property
    def pn(self) -> float: return math.pi * self.mn

    @property
    def pt(self) -> float: return math.pi * self.mt

    @property
    def pbt(self) -> float:
        "Transverse base pitch [mm]."
        return self.pt * math.cos(self.alpha_t)

    @property
    def pbn(self) -> float:
        "Normal base pitch [mm]."
        return self.pn * math.cos(self.alpha_n)

    @property
    def px(self) -> float:
        "Axial pitch [mm] (infinite for a spur gear)."
        return math.inf if self.beta == 0 else self.pn / math.sin(self.beta)

    @property
    def lead(self) -> float:
        "Lead p_z [mm]: axial advance of a tooth over one turn (infinite for a spur gear)."
        return math.inf if self.beta == 0 else math.pi * self.d / math.tan(self.beta)

    def beta_at(self, dy: float) -> float:
        "Helix angle [rad] on diameter dy [mm]."
        return math.atan(math.tan(self.beta) * dy / self.d)

    # --- thickness -------------------------------------------------------------------
    @property
    def sn(self) -> float:
        "Normal arc tooth thickness on the reference circle [mm], no allowance."
        return self.mn * (math.pi / 2 + 2 * self.x * math.tan(self.alpha_n))

    @property
    def st(self) -> float:
        "Transverse arc tooth thickness on the reference circle [mm], no allowance."
        return self.sn / math.cos(self.beta)

    @property
    def san(self) -> float:
        "Normal tooth thickness on the tip circle [mm]."
        return self.transverse().sa * math.cos(self.beta_at(self.da))

    @property
    def zn(self) -> float:
        "Virtual tooth count z / (cos² β_b cos β) for strength (ISO 6336-3)."
        return self.z / (math.cos(self.beta_b) ** 2 * math.cos(self.beta))

    # --- equivalent spur gears -------------------------------------------------------
    def transverse(self) -> GearGeometry:
        "The equivalent transverse spur gear: same diameters, flanks and root form (see above)."
        cb = math.cos(self.beta)
        rho_t = self.rho_f * cb * (1 - math.sin(self.alpha_n)) / (1 - math.sin(self.alpha_t))
        return GearGeometry(z=self.z, m=self.mt, x=self.x * cb, b=self.b, alpha=self.alpha_t,
                            ha=self.ha * cb, hf=self.hf * cb, rho_f=rho_t, k_tip=self.k_tip * cb, name=self.name)

    def virtual(self) -> GearGeometry:
        "The virtual spur gear (z_n teeth, normal module and pressure angle) for tooth strength."
        return GearGeometry(z=self.zn, m=self.mn, x=self.x, b=self.b, alpha=self.alpha_n,  # type: ignore[arg-type]
                            ha=self.ha, hf=self.hf, rho_f=self.rho_f, k_tip=self.k_tip, name=self.name)

    @property
    def d_form(self) -> float:
        "Root form diameter d_Ff of the hobbed gear [mm]."
        return self.transverse().d_form

    @property
    def z_min_undercut(self) -> float:
        "Smallest tooth count without undercut for this shift and helix angle."
        return self.transverse().z_min_undercut

    @property
    def undercut(self) -> bool:
        return self.transverse().undercut

    def with_(self, **kw) -> "HelicalGear":
        "A copy with some fields changed."
        return replace(self, **kw)

In [ ]:
g = HelicalGear(z=20, mn=2.0, beta=math.radians(15), x=0.2, b=20, name="pinion")
print(f"m_t = {g.mt:.4f}, α_t = {math.degrees(g.alpha_t):.4f}°, β_b = {math.degrees(g.beta_b):.4f}°")
print(f"d = {g.d:.4f}, d_b = {g.db:.4f}, d_a = {g.da:.4f}, d_f = {g.df:.4f}, d_Ff = {g.d_form:.4f}")
print(f"p_x = {g.px:.3f}, lead = {g.lead:.2f}, z_n = {g.zn:.2f}, s_n = {g.sn:.4f}, s_an = {g.san:.3f}")

In [ ]:
#| hide
# transverse equivalent keeps every length
_t = g.transverse()
assert math.isclose(_t.d, g.d) and math.isclose(_t.da, g.da) and math.isclose(_t.df, g.df)
assert math.isclose(_t.s, g.st, rel_tol=1e-12)
# tan βb = tan β cos αt  ⇔  sin βb = sin β cos αn
assert math.isclose(math.tan(g.beta_b), math.tan(g.beta) * math.cos(g.alpha_t), rel_tol=1e-12)
assert math.isclose(g.beta_at(g.db), g.beta_b, rel_tol=1e-12)
# mn cos αn = mt cos αt cos βb (normal and transverse base pitch)
assert math.isclose(g.pbn, g.pbt * math.cos(g.beta_b), rel_tol=1e-12)
# β = 0 reduces to the spur gear
_s = HelicalGear(z=20, mn=2.0, beta=0.0, x=0.2, b=20)
_sp = GearGeometry(z=20, m=2.0, x=0.2, b=20)
for _a in ("d", "db", "da", "df", "d_form", "z_min_undercut"):
    assert math.isclose(getattr(_s.transverse(), _a), getattr(_sp, _a), rel_tol=1e-12), _a
assert math.isclose(_s.zn, 20) and _s.px == math.inf
# undercut limit (closed form above)
_b, _an = math.radians(15), math.radians(20)
_at = math.atan(math.tan(_an) / math.cos(_b))
_zmin = 2 * math.cos(_b) * (1.25 - 0.0 - 0.38 * (1 - math.sin(_an))) / math.sin(_at) ** 2
assert math.isclose(HelicalGear(z=14, mn=1, beta=_b).z_min_undercut, _zmin, rel_tol=1e-12)
for _bad in (dict(z=2), dict(mn=0), dict(beta=math.radians(46)), dict(hand="up")):
    try:
        HelicalGear(**{**dict(z=20, mn=1.0, beta=0.2), **_bad})
        raise AssertionError(_bad)
    except ValueError:
        pass

## A Meshing Pair

Two external helical gears mesh when they have the same normal module, the same normal pressure angle, the same helix angle and **opposite hands**. In the transverse section they are a spur pair with module $m_t$ and pressure angle $\alpha_t$. The operating transverse pressure angle and the centre distance are (ISO 21771)

$$\operatorname{inv}\alpha_{wt} = \operatorname{inv}\alpha_t + 2\tan\alpha_n\,\frac{x_1 + x_2}{z_1 + z_2},\qquad
a_w = \frac{(z_1+z_2)\,m_t}{2}\,\frac{\cos\alpha_t}{\cos\alpha_{wt}} .$$

These are the spur formulas applied to the transverse equivalents, because $x\cos\beta\tan\alpha_t = x\tan\alpha_n$. The inverse problem, the shift sum for a given centre distance, is

$$x_1 + x_2 = \frac{(z_1+z_2)\,(\operatorname{inv}\alpha_{wt} - \operatorname{inv}\alpha_t)}{2\tan\alpha_n},\qquad \cos\alpha_{wt} = \frac{a\cos\alpha_t}{a_w},$$

with $a = (z_1+z_2)m_t/2$. Unlike a spur pair, a helical pair has a second way to reach a centre distance: the helix angle. With zero shift sum,

$$\cos\beta = \frac{m_n(z_1+z_2)}{2a} .$$

`gears.helical.layout` uses both. The operating helix angle on the pitch cylinder is $\tan\beta_w = \tan\beta\,\cos\alpha_t/\cos\alpha_{wt}$, and it sets the axial force.

In [ ]:
#| export
def _check_helical_pair(g1: HelicalGear, g2: HelicalGear):
    if not math.isclose(g1.mn, g2.mn) or not math.isclose(g1.alpha_n, g2.alpha_n):
        raise ValueError("gears of a pair need the same normal module and normal pressure angle")
    if not math.isclose(g1.beta, g2.beta, abs_tol=1e-12):
        raise ValueError("gears of an external pair need the same helix angle")
    if g1.beta > 0 and g1.hand == g2.hand:
        raise ValueError("gears of an external helical pair need opposite hands")


def operating_transverse_angle(g1: HelicalGear, g2: HelicalGear) -> float:
    "Operating transverse pressure angle α_wt [rad] of a backlash-free external pair."
    _check_helical_pair(g1, g2)
    return operating_pressure_angle(g1.transverse(), g2.transverse())


def helical_centre_distance(g1: HelicalGear, g2: HelicalGear) -> float:
    "Operating centre distance a_w [mm] of a backlash-free external pair."
    _check_helical_pair(g1, g2)
    return centre_distance(g1.transverse(), g2.transverse())


def helical_shift_sum(z1: int, z2: int, mn: float, beta: float, a_w: float,
                      alpha_n: float = math.radians(20.0)) -> float:
    "Profile shift sum x1 + x2 (normal modules) that gives centre distance a_w [mm]."
    at = math.atan(math.tan(alpha_n) / math.cos(beta))
    a = (z1 + z2) * mn / math.cos(beta) / 2
    c = a * math.cos(at) / a_w
    if not 0 < c < 1:
        raise ValueError("centre distance not reachable with an involute mesh")
    awt = math.acos(c)
    return (z1 + z2) * (float(inv(awt)) - float(inv(at))) / (2 * math.tan(alpha_n))


def helix_for_centre_distance(z1: int, z2: int, mn: float, a: float) -> float:
    "Helix angle β [rad] that gives reference centre distance a [mm] with zero shift sum."
    c = mn * (z1 + z2) / (2 * a)
    if not 0 < c <= 1:
        raise ValueError("centre distance smaller than the spur reference centre distance")
    return math.acos(c)


def overlap_ratio(b: float, mn: float, beta: float) -> float:
    "Overlap ratio ε_β = b sin β / (π m_n)."
    return b * math.sin(beta) / (math.pi * mn)

## Contact Ratios

In the transverse section the pair has the transverse contact ratio $\varepsilon_\alpha$ of its spur equivalents: the length of the path of contact divided by the transverse base pitch,

$$\varepsilon_\alpha = \frac{\sqrt{r_{a1}^2 - r_{b1}^2} + \sqrt{r_{a2}^2 - r_{b2}^2} - a_w\sin\alpha_{wt}}{p_{bt}} .$$

Along the face, the teeth are twisted by one axial pitch every $p_x$ of width. So each contact line is spread over $b/p_x$ more pitches, which gives the **overlap ratio**

$$\varepsilon_\beta = \frac{b}{p_x} = \frac{b\sin\beta}{\pi m_n},\qquad \varepsilon_\gamma = \varepsilon_\alpha + \varepsilon_\beta .$$

The total contact ratio $\varepsilon_\gamma$ is the average number of tooth pairs in contact. Overlap is why a helical pair runs quieter than a spur pair. Contact starts gradually across the face instead of along a whole line at once. The mesh stiffness also varies less from one engagement to the next, and for an integer $\varepsilon_\beta$ the total length of the contact lines stays constant, which minimises the excitation.

The checks follow `gears.involute.mesh_check` on the transverse equivalents: the active profile must stay on the involute (no tip–fillet interference) and both tip clearances must be positive. The transverse contact ratio may be lower than for a spur gear because the overlap carries the load across. Here the limit is $\varepsilon_\alpha \ge 1.1$ when $\varepsilon_\beta \ge 1$, and $1.2$ otherwise.

In [ ]:
#| export
@dataclass(frozen=True)
class HelicalMesh:
    """
    Geometry checks of an external helical mesh. Lengths in mm, angles in radians.

    alpha_wt      : operating transverse pressure angle
    beta_w        : operating helix angle (on the pitch cylinders)
    a_w           : operating centre distance
    eps_alpha     : transverse contact ratio
    eps_beta      : overlap ratio (narrower face width)
    eps_gamma     : total contact ratio
    d_Nf1, d_Nf2  : start of the active profile on each gear
    d_Ff1, d_Ff2  : root form diameters
    clearance12   : gear 1 tip to gear 2 root
    clearance21   : gear 2 tip to gear 1 root
    interference  : contact runs into a fillet
    """
    alpha_wt: float
    beta_w: float
    a_w: float
    eps_alpha: float
    eps_beta: float
    eps_gamma: float
    d_Nf1: float
    d_Nf2: float
    d_Ff1: float
    d_Ff2: float
    clearance12: float
    clearance21: float
    interference: bool

    @property
    def eps_alpha_min(self) -> float:
        "Smallest acceptable transverse contact ratio (see above)."
        return 1.1 if self.eps_beta >= 1 else 1.2

    @property
    def ok(self) -> bool:
        return (self.eps_alpha >= self.eps_alpha_min and not self.interference
                and min(self.clearance12, self.clearance21) > 0)


def helical_mesh_check(g1: HelicalGear, g2: HelicalGear) -> HelicalMesh:
    "Contact ratios, active profile, interference and tip clearance of an external helical pair."
    _check_helical_pair(g1, g2)
    t1, t2 = g1.transverse(), g2.transverse()
    mc = mesh_check(t1, t2)
    eb = overlap_ratio(min(g1.b, g2.b), g1.mn, g1.beta)
    beta_w = math.atan(math.tan(g1.beta) * math.cos(g1.alpha_t) / math.cos(mc.alpha_w))
    return HelicalMesh(mc.alpha_w, beta_w, mc.a_w, mc.eps_alpha, eb, mc.eps_alpha + eb, mc.d_Nf1, mc.d_Nf2,
                       mc.d_Ff1, mc.d_Ff2, mc.clearance12, mc.clearance21, mc.interference)

A worked example from the KHK gear handbook (normal system helical gears): $m_n = 3$, $\alpha_n = 20°$, $\beta = 30°$, $z_1 = 12$, $z_2 = 60$, $x_1 = 0.09809$, $x_2 = 0$ gives $a_w = 125$ mm and $\alpha_{wt} = 23.1126°$.

In [ ]:
p = HelicalGear(z=12, mn=3, beta=math.radians(30), x=0.09809, b=30, hand="right", name="pinion")
w = HelicalGear(z=60, mn=3, beta=math.radians(30), x=0.0, b=30, hand="left", name="wheel")
mc = helical_mesh_check(p, w)
print(f"a_w = {mc.a_w:.4f} mm, α_wt = {math.degrees(mc.alpha_wt):.4f}°, β_w = {math.degrees(mc.beta_w):.3f}°")
print(f"ε_α = {mc.eps_alpha:.3f}, ε_β = {mc.eps_beta:.3f}, ε_γ = {mc.eps_gamma:.3f}, ok = {mc.ok}")

In [ ]:
#| hide
assert abs(mc.a_w - 125.0) < 2e-3
assert abs(math.degrees(mc.alpha_wt) - 23.1126) < 2e-4
assert abs(helical_shift_sum(12, 60, 3, math.radians(30), 125.0) - 0.09809) < 2e-4
assert math.isclose(mc.eps_beta, 30 * math.sin(math.radians(30)) / (math.pi * 3))
assert math.isclose(math.degrees(helix_for_centre_distance(20, 60, 2.0, 82.0)),
                    math.degrees(math.acos(80 / 82)))
for _bad in (w.with_(hand="right"), w.with_(beta=0.5), w.with_(mn=2.5)):
    try:
        helical_mesh_check(p, _bad)
        raise AssertionError("pair check missed a mismatch")
    except ValueError:
        pass
# a shift sum found from a centre distance reproduces it
_xs = helical_shift_sum(17, 52, 1.5, math.radians(18), 55.0)
_a = helical_centre_distance(HelicalGear(17, 1.5, math.radians(18), _xs / 2, hand="right"),
                             HelicalGear(52, 1.5, math.radians(18), _xs / 2, hand="left"))
assert abs(_a - 55.0) < 1e-9

In [ ]:
#| echo: false
def _contact_figure():
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for bm in (6, 10, 15):
        betas = np.radians(np.linspace(0, 35, 71))
        ax[0].plot(np.degrees(betas), [overlap_ratio(bm * 1.0, 1.0, b_) for b_ in betas], label=f"b = {bm} m_n")
    ax[0].axhline(1, color="#888", lw=0.8, ls="--")
    ax[0].set_xlabel("helix angle β [°]"); ax[0].set_ylabel("overlap ratio ε_β"); ax[0].legend(frameon=False)
    ax[0].set_title("Overlap grows with face width and helix angle", fontsize=10)
    betas = np.radians(np.linspace(0, 35, 36))
    ea = []
    for b_ in betas:
        g1 = HelicalGear(20, 1.0, b_, 0.0, hand="right")
        g2 = HelicalGear(60, 1.0, b_, 0.0, hand="left")
        ea.append(helical_mesh_check(g1, g2).eps_alpha)
    ax[1].plot(np.degrees(betas), ea, color="#b8322a")
    ax[1].set_xlabel("helix angle β [°]"); ax[1].set_ylabel("transverse contact ratio ε_α")
    ax[1].set_title("…while ε_α falls (20/60 teeth, x = 0)", fontsize=10)
    for a_ in ax:
        a_.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
_contact_figure()

## Inspection Dimensions in the Normal Section

### Span measurement

A disc micrometer measures the span $W_k$ across $k$ teeth along a line tangent to the base cylinder, in the **normal** plane of the base helix. In the transverse section the base tangent length over $k$ teeth is $(k-1)p_{bt} + s_{bt}$. Projected onto the normal of the base helix it shortens by $\cos\beta_b$, which gives (ISO 21771, DIN 3960)

$$W_k = m_n\cos\alpha_n\left[(k-1)\pi + z\operatorname{inv}\alpha_t\right] + s_n\cos\alpha_n .$$

With the nominal $s_n$ this is the familiar $m_n\cos\alpha_n[(k-0.5)\pi + z\operatorname{inv}\alpha_t] + 2xm_n\sin\alpha_n$. The span count puts the contact near mid-depth:

$$k = \operatorname{round}\!\left[\frac{z}{\pi}\left(\frac{\tan\alpha_x}{\cos^2\beta_b} - \frac{2x\tan\alpha_n}{z} - \operatorname{inv}\alpha_t\right) + 0.5\right],\qquad \cos\alpha_x = \frac{d_b}{d + 2xm_n} .$$

The disc touches the flanks at two points that are $W_k\sin\beta_b$ apart along the axis. So the face must be wider than that, plus room for the discs. `span_fits` asks for $b \ge W_k\sin\beta_b + 2$ mm. Below that, use the ball measurement.

In [ ]:
#| export
def helical_span_teeth(g: HelicalGear) -> int:
    "Number of teeth k for the span measurement of a helical gear."
    ax = math.acos(min(1.0, g.db / (g.d + 2 * g.x * g.mn)))
    k = round(g.z / math.pi * (math.tan(ax) / math.cos(g.beta_b) ** 2 - 2 * g.x * math.tan(g.alpha_n) / g.z
                               - float(inv(g.alpha_t))) + 0.5)
    return int(min(max(k, 2), g.z - 1))


def helical_span_measurement(g: HelicalGear, sn: float | None = None, k: int | None = None) -> float:
    "Span W_k [mm] over k teeth in the normal section, for normal reference tooth thickness sn (default nominal)."
    k = helical_span_teeth(g) if k is None else k
    sn = g.sn if sn is None else sn
    ca = math.cos(g.alpha_n)
    return ca * (g.mn * ((k - 1) * math.pi + g.z * float(inv(g.alpha_t))) + sn)


def span_fits(g: HelicalGear, k: int | None = None, margin: float = 2.0) -> bool:
    "True if the face is wide enough for a disc micrometer: b ≥ W_k sin β_b + margin."
    return g.b >= helical_span_measurement(g, k=k) * math.sin(g.beta_b) + margin

### Dimension over balls

Two balls of diameter $D_M$ placed in opposite tooth spaces, in the same transverse plane, touch the flanks along the normal of the base helix. In the transverse section the ball acts like a pin of diameter $D_M/\cos\beta_b$. So the pin formula of the spur module applies with that diameter (DIN 3960):

$$\operatorname{inv}\alpha_{Mt} = \frac{s_t}{d} + \operatorname{inv}\alpha_t + \frac{D_M}{d_b\cos\beta_b} - \frac{\pi}{z},\qquad
M_d = \frac{d_b}{\cos\alpha_{Mt}}\,\left[\cos\frac{\pi}{2z}\right]_{z\text{ odd}} + D_M .$$

Here $d_b\cos\beta_b = z\,m_n\cos\alpha_n$, so the term is the familiar $D_M/(z\,m_n\cos\alpha_n)$. The ball size is chosen as in the spur module. The ball touches near the reference circle and is enlarged until it stands at least $0.2\,m_n$ above the tips. The returned contact diameter is the transverse estimate. The true contact point lies slightly off the ball's transverse plane.

In [ ]:
#| export
def helical_ball_measurement(g: HelicalGear, D: float | None = None, sn: float | None = None) -> tuple[float, float]:
    "Dimension over two balls M_d [mm] and the (approximate) contact diameter, for normal thickness sn."
    D = helical_ball_diameter(g) if D is None else D
    sn = g.sn if sn is None else sn
    st = sn / math.cos(g.beta)
    aM = inv_inverse(st / g.d + float(inv(g.alpha_t)) + D / (g.db * math.cos(g.beta_b)) - math.pi / g.z)
    factor = 1.0 if g.z % 2 == 0 else math.cos(math.pi / (2 * g.z))
    M = g.db * factor / math.cos(aM) + D
    tan_c = math.tan(aM) - D / (g.db * math.cos(g.beta_b))
    return M, g.db / math.cos(math.atan(tan_c))


def helical_ball_diameter(g: HelicalGear, protrude: float = 0.2) -> float:
    "Ball diameter [mm], rounded to 0.01 mm: touches near the reference circle and stands `protrude`·m_n above the tips."
    eta = math.pi / g.z - g.st / g.d
    D = round(g.db * (math.tan(g.alpha_t + eta) - math.tan(g.alpha_t)) * math.cos(g.beta_b), 2)
    for _ in range(400):
        M, _ = helical_ball_measurement(g, D)
        if M - g.da >= protrude * g.mn:
            break
        D = round(D + 0.01, 2)
    return D

In [ ]:
g = HelicalGear(z=23, mn=2.0, beta=math.radians(20), x=0.3, b=30, name="pinion")
k = helical_span_teeth(g)
print(f"W_{k} = {helical_span_measurement(g):.4f} mm (fits: {span_fits(g)}), "
      f"M_d = {helical_ball_measurement(g)[0]:.4f} mm over Ø{helical_ball_diameter(g):.2f} balls")

In [ ]:
#| hide
from emachines.gears.involute import span_measurement, span_teeth, ball_measurement
# normal span = transverse base tangent length × cos βb
_t = g.transverse()
_Wt = (k - 1) * _t.pb + _t.db * (_t.s / _t.d + float(inv(_t.alpha)))
assert math.isclose(helical_span_measurement(g), _Wt * math.cos(g.beta_b), rel_tol=1e-12)
# textbook form with the nominal thickness
_W = g.mn * math.cos(g.alpha_n) * ((k - 0.5) * math.pi + g.z * float(inv(g.alpha_t))) + 2 * g.x * g.mn * math.sin(g.alpha_n)
assert math.isclose(helical_span_measurement(g), _W, rel_tol=1e-12)
# β = 0 reproduces the spur module
_s = HelicalGear(z=23, mn=2.0, beta=0.0, x=0.3, b=30)
_sp = GearGeometry(z=23, m=2.0, x=0.3, b=30)
assert helical_span_teeth(_s) == span_teeth(_sp)
assert math.isclose(helical_span_measurement(_s), span_measurement(_sp), rel_tol=1e-12)
assert math.isclose(helical_ball_measurement(_s, 3.5)[0], ball_measurement(_sp, 3.5)[0], rel_tol=1e-12)
# thinner teeth → smaller span and smaller dimension over balls; balls stand out past the tips
assert helical_span_measurement(g, g.sn - 0.05) < helical_span_measurement(g)
assert helical_ball_measurement(g, sn=g.sn - 0.05)[0] < helical_ball_measurement(g)[0]
assert helical_ball_measurement(g)[0] - g.da >= 0.2 * g.mn - 1e-9

## References

1. ISO 21771:2007, *Gears — Cylindrical involute gears and gear pairs — Concepts and geometry* (transverse and normal quantities, centre distance, contact ratios, span and ball measurements).
2. DIN 3960:1987, *Definitions, parameters and equations for involute cylindrical gears and gear pairs* (measurement over balls of helical gears).
3. ISO 6336-3:2006, *Calculation of load capacity of spur and helical gears — Part 3: Tooth bending strength* (virtual tooth count $z_n$).
4. KHK Stock Gears, *Gear Technical Reference*, normal system helical gear calculation example.
5. G. Niemann, H. Winter, *Maschinenelemente*, Band II, Springer, 2nd ed., 1989, Ch. 21–22.